# Clustering por atributos y análisis espacial para Siniestros fatales del Perú

**Pregunta:** ¿qué perfiles de siniestros fatales aparecen según la gravedad y las condiciones vehiculares, y dónde se localizan?

## 1. K - Means (seleccionaremos 10 variables de la matriz analítica)

Se eligió **K-Means** porque permite comparar 9 106 siniestros con variables numéricas preparadas, formar grupos reproducibles e interpretar sus centroides. Frente al **clustering jerárquico aglomerativo**, es más sencillo de ejecutar con esta cantidad de registros; frente a **DBSCAN espacial**, agrupa por características del siniestro y no por cercanía entre coordenadas. Su inconveniente es que depende de la escala, de los atributos y del número de grupos elegido.

| Variables de entrada (10) | Justificación |
|---|---|
| `cantidad_de_fallecidos`, `cantidad_de_lesionados` | Describen las consecuencias humanas del siniestro. |
| `n_vehiculos` | Distingue eventos según los vehículos involucrados. |
| `edad_promedio` | Incorpora un resumen de la edad registrada. |
| `prop_seguro`, `prop_citv`, `prop_fugados` | Comparan la proporción de vehículos asegurados, con CITV vigente o fugados, incluso si el siniestro incluye varios vehículos. |
| `existe_moto`, `existe_conductor_sin_licencia_vigente`, `existe_dosaje_cualitativo_positivo` | Indican condiciones registradas de participación vehicular y conductores (0 = no, 1 = sí). |

**Por qué dejamos fuera el resto:**

- **`clase_*` y `causa_*`:** queremos contrastar los perfiles con las categorías oficiales, no formar grupos que simplemente las reproduzcan. Muchas causas siguen en investigación.
- **`coordenadas_latitud` y `coordenadas_longitud`:** se utilizan **después**, para mapear los perfiles; así la distancia geográfica no modifica la similitud de atributos.
- **`zona_*` y `peaton_involucrado`:** se reservan para describir el contexto rural/urbano y la relación con peatones.
- **`hora_siniestro`, `trimestre` y `red_vial`:** son códigos o categorías; tratarlos directamente como números introduciría distancias artificiales. Se examinan luego en tablas.
- **`n_personas`, `n_masculino` y `n_femenino`:** contienen información parcialmente redundante y algunas inconsistencias de registro.
- **Conteos brutos de seguro, CITV y fuga:** se reemplazan por **proporciones** para no privilegiar automáticamente siniestros con más vehículos.

Estas exclusiones son **decisiones metodológicas**, no significan que las variables carezcan de interés. Las recuperamos tras el agrupamiento para caracterizar y contrastar los resultados.

In [ ]:
%matplotlib inline
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import folium
from folium.plugins import FastMarkerCluster
from branca.colormap import linear
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.neighbors import NearestNeighbors

# Archivo en la misma carpeta que este notebook.
df = pd.read_excel("matriz_analit.xlsx", index_col=0)
print("Matriz:", df.shape)
display(df[["cantidad_de_fallecidos", "cantidad_de_lesionados", "n_vehiculos", "edad_promedio"]].head(3))

## 2. Preparación y transformación de variables

Partimos de los cuatro conteos/promedios originales, tres indicadores binarios y tres proporciones derivadas (esto para ahorrarnos de cierta forma variables redundantes):

$
 p_{seguro}=\frac{n_{con\ seguro\ vigente}}{n_{vehículos}},\qquad
 p_{CITV}=\frac{n_{con\ CITV\ vigente}}{n_{vehículos}},\qquad
 p_{fugados}=\frac{n_{vehiculos\ fugados}}{n_{vehículos}}.
$

En la matriz, cada siniestro tiene al menos un vehículo. Estas proporciones están entre 0 y 1: describen la **fracción de vehículos por siniestro**, no el porcentaje global de siniestros.

Luego aplicamos las siguientes transformaciones:

1. **Edad:** en dos registros `edad_promedio = 0` se sustituye por la mediana de edades positivas (**38 años**), porque se considera un valor no informativo.
2. **Fallecidos y lesionados:** usamos $x'=\ln(1+x)$ (`np.log1p`) para reducir la influencia de valores excepcionalmente altos, manteniendo el orden de los registros.
3. **Siete atributos numéricos:** aplicamos `StandardScaler`, es decir, $z=(x-\bar{x})/s$. Así se comparan sin que dominen sus unidades originales; la estandarización se realiza **después** del logaritmo.
4. **Tres indicadores binarios:** permanecen en **0/1**; no se estandarizan. Esto conserva una interpretación directa, aunque su peso relativo frente a las variables estandarizadas depende de las distancias euclidianas.

La matriz de entrada `X` tiene finalmente **9 106 filas y 10 columnas**. Las versiones originales permanecen en `df` para interpretar los resultados.

In [ ]:
numericas = ["cantidad_de_fallecidos", "cantidad_de_lesionados", "n_vehiculos",
             "edad_promedio", "prop_seguro", "prop_citv", "prop_fugados"]
binarias = ["existe_moto", "existe_conductor_sin_licencia_vigente",
            "existe_dosaje_cualitativo_positivo"]

X = df[["cantidad_de_fallecidos", "cantidad_de_lesionados", "n_vehiculos",
        "edad_promedio"] + binarias].copy()
X["prop_seguro"] = df["n_con_seguro_vigente"] / df["n_vehiculos"]
X["prop_citv"] = df["n_con_citv_vigente"] / df["n_vehiculos"]
X["prop_fugados"] = df["n_vehiculos_fugados"] / df["n_vehiculos"]
X = X[numericas + binarias]

X.loc[X["edad_promedio"] == 0, "edad_promedio"] = X.loc[
    X["edad_promedio"] > 0, "edad_promedio"].median()
for columna in ["cantidad_de_fallecidos", "cantidad_de_lesionados"]:
    X[columna] = np.log1p(X[columna])
X[numericas] = StandardScaler().fit_transform(X[numericas])
print("Variables utilizadas:", X.shape[1], "| Registros:", len(X))

## 3. Evaluación de K y entrenamiento

Comparamos **inercia** (menor), **Silhouette** (mayor) y **Davies–Bouldin** (menor). La inercia disminuye al aumentar K, por lo que no basta por sí sola. Se mantiene **K = 4** para reproducir el análisis original y sus perfiles interpretables; no significa que cuatro grupos sean el único resultado posible.

Usamos `k-means++`, `n_init=10` y `random_state=42`, las mismas configuraciones del notebook de referencia.

In [ ]:
evaluacion = []
for k in range(2, 8):
    m = KMeans(n_clusters=k, init="k-means++", n_init=10, random_state=42)
    etiquetas = m.fit_predict(X)
    evaluacion.append([k, m.inertia_,
                       silhouette_score(X, etiquetas, sample_size=1500, random_state=42),
                       davies_bouldin_score(X, etiquetas)])

evaluacion = pd.DataFrame(evaluacion, columns=["K", "Inercia", "Silueta", "Davies-Bouldin"])
display(evaluacion.round(3))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, metrica, titulo in zip(axes,
                               ["Inercia", "Silueta", "Davies-Bouldin"],
                               ["Método del codo", "Silhouette", "Davies–Bouldin"]):
    ax.plot(evaluacion["K"], evaluacion[metrica], "o-")
    ax.axvline(x=4, color="red", linestyle="--", linewidth=1.5, label="K = 4")
    ax.set(xlabel="K", ylabel=metrica, title=titulo)
    ax.set_xticks(evaluacion["K"])
    ax.grid(alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
modelo = KMeans(n_clusters=4, init="k-means++", n_init=10, random_state=42)
df["cluster_kmeans"] = modelo.fit_predict(X)
print("Inercia con K=4:", round(modelo.inertia_, 2))
display(df["cluster_kmeans"].value_counts().sort_index().rename("Siniestros").to_frame())

In [ ]:

import matplotlib.pyplot as plt
import seaborn as sns

# Promedios estandarizados de las 7 variables numéricas
perfil_std = X[numericas].copy()
perfil_std["cluster"] = df["cluster_kmeans"].to_numpy()

heatmap_clusters = (
    perfil_std.groupby("cluster")[numericas]
    .mean()
    .T
)

# Crear el heatmap
plt.figure(figsize=(8, 5))

sns.heatmap(
    heatmap_clusters,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    linewidths=0.5,
    cbar_kws={"label": "Promedio estandarizado"}
)

plt.title("Características promedio por clúster (K=4)")
plt.xlabel("Clúster")
plt.ylabel("Variables")
plt.tight_layout()

# Guardar imagen para el informe
plt.savefig("heatmap_clusters.png", dpi=300, bbox_inches="tight")
plt.show()

print("Imagen guardada: heatmap_clusters.png")


### Interpretación del heatmap

El clúster 0 destaca por una mayor cantidad de fallecidos y lesionados, mientras que el clúster 1 presenta una mayor proporción de vehículos fugados. Por otro lado, el clúster 2 muestra valores más altos de seguro y CITV vigentes, y el clúster 3 presenta valores menores en estos indicadores.

En general, el gráfico ayuda a identificar las características que diferencian a los cuatro grupos.

## 4. Interpretación de los perfiles usando los atributos del modelo

In [ ]:
perfil_datos = df.assign(
    prop_seguro=df["n_con_seguro_vigente"] / df["n_vehiculos"],
    prop_citv=df["n_con_citv_vigente"] / df["n_vehiculos"],
    prop_fugados=df["n_vehiculos_fugados"] / df["n_vehiculos"])
perfil = perfil_datos.groupby("cluster_kmeans").agg(
    siniestros=("cluster_kmeans", "size"),
    fallecidos_prom=("cantidad_de_fallecidos", "mean"),
    lesionados_prom=("cantidad_de_lesionados", "mean"),
    vehiculos_prom=("n_vehiculos", "mean"),
    edad_prom=("edad_promedio", "mean"),
    seguro_pct=("prop_seguro", "mean"),
    citv_pct=("prop_citv", "mean"),
    fugados_pct=("prop_fugados", "mean"),
    moto_pct=("existe_moto", "mean"),
    sin_licencia_pct=("existe_conductor_sin_licencia_vigente", "mean"),
    dosaje_pct=("existe_dosaje_cualitativo_positivo", "mean"))
perfil.loc[:, perfil.columns.str.endswith("_pct")] *= 100
display(perfil.round(1))

### ¿Qué representa cada grupo según los diez atributos?

- **G0 — Mayor gravedad humana (1 165; 12,8 %):** registra **2,22 fallecidos** y **4,79 lesionados por siniestro**, además de **1,64 vehículos** y **7,64 personas** involucradas, en promedio. La gravedad elevada **no es un hallazgo independiente**: fallecidos y lesionados participaron en el entrenamiento.
- **G1 — Fuga vehicular (885; 9,7 %):** tiene una **proporción media de vehículos fugados del 98,8 %**. Promedia **1,01 fallecidos**, **0,03 lesionados** y **1,03 vehículos** por siniestro. La cobertura registrada de seguro y CITV es casi nula, posiblemente afectada por datos no identificados de vehículos fugados.
- **G2 — Mayor cobertura vehicular (3 510; 38,5 %):** alcanza **97,1 % de seguro** y **75,7 % de CITV vigente**, como promedios de proporciones; registra **1,05 fallecidos** y **0,24 lesionados** por siniestro. Esto describe cobertura documentada, **no** que sean accidentes «seguros».
- **G3 — Motocicletas y conductores sin licencia (3 546; 38,9 %):** **59,1 %** de siniestros involucra motos y **53,9 %** registra algún conductor sin licencia vigente; presenta **34,4 % de seguro** y **4,4 % de CITV** como proporciones medias, además de **8,3 %** con dosaje positivo registrado. Los indicadores de moto y licencia no implican que ocurran simultáneamente en todos los siniestros.

## 5. Añadimos variables excluidas **sin volver a entrenar**

Ahora contrastamos los perfiles con **clase del accidente, causa, zona, peatones, demografía y códigos de contexto**. Como esas columnas no entraron en K-means, sus asociaciones aportan información adicional: permiten comprobar **qué clase de siniestro corresponde a cada perfil** y **en qué contexto aparece**.

In [ ]:
excluidas = ["zona_RURAL", "zona_URBANA", "peaton_involucrado",
             "clase_ATROPELLO", "clase_CHOQUE", "clase_DESPISTE",
             "causa_INVESTIGACION", "causa_CONDUCTOR"]
contexto = df.groupby("cluster_kmeans")[excluidas].mean().mul(100)
contexto["personas_prom"] = df.groupby("cluster_kmeans")["n_personas"].mean()
contexto["hombres_prom"] = df.groupby("cluster_kmeans")["n_masculino"].mean()
contexto["mujeres_prom"] = df.groupby("cluster_kmeans")["n_femenino"].mean()
display(contexto.round(1))

# Estas categorías se muestran como distribuciones; sus números no son magnitudes continuas.

#for variable in ["hora_siniestro", "trimestre", "red_vial"]:
#    print("Distribución (%) de", variable)
#    display((pd.crosstab(df["cluster_kmeans"], df[variable], normalize="index") * 100).round(1))

### Variables de contexto no usadas para agrupar

También podemos comparar los códigos de hora, trimestre y red vial **sin tratarlos como distancias numéricas**. La interpretación específica de sus valores requiere el diccionario de datos.

In [ ]:
for variable in ["hora_siniestro", "trimestre", "red_vial"]:
    print(f"Distribución de {variable} por clúster (%)")
    display((pd.crosstab(df["cluster_kmeans"], df[variable], normalize="index") * 100).round(1))

### Interpretación final tras incorporar las variables excluidas

**G0 — Perfil de alta gravedad, principalmente choques y despistes rurales.** 

El **82,5 %** de sus registros corresponde a zona rural; predominan **choques (50,6 %)** y **despistes (44,5 %)**, mientras que los atropellos solo representan **1,2 %**. El **53,8 %** tiene causa asociada al conductor y **44,0 %** conserva causa en investigación. Aunque reúne apenas el **12,8 %** de los siniestros, acumula aproximadamente **23,9 % de los fallecidos** y **71,3 % de los lesionados** de la matriz.

**G1 — Perfil de fuga vehicular asociado a atropellos.** 

Al recuperar las clases vemos que **92,2 %** son **atropellos** y **92,3 %** registran **peatones involucrados**, pese a que ambas variables fueron excluidas de K-means. También, **76,6 %** conserva la causa en investigación; los siniestros se dividen casi por igual entre zonas urbanas y rurales (**49,8 %** en cada una). Esta asociación entre fuga y atropello es uno de los hallazgos más claros. Una baja cobertura de seguro/CITV registrada puede reflejar **falta de datos** de vehículos fugados, no ausencia demostrada de esos requisitos.

**G2 — Perfil de mayor cobertura vehicular, sin clase dominante.** 

Reúne **choques (36,2 %)**, **atropellos (35,5 %)** y **despistes (23,1 %)**; el **38,2 %** registra peatones involucrados y **52,4 %** ocurre en zona rural. El hallazgo es que una cobertura documental relativamente alta **atraviesa varias clases** de accidentes. No permite afirmar que estos vehículos o conductores fueron más responsables o menos riesgosos.

**G3 — Perfil de motos/licencias, principalmente choques y despistes.** 

Las clases predominantes son **choques (38,7 %)** y **despistes (34,5 %)**, seguidas de **atropellos (19,2 %)**. El **55,3 %** aparece en zona rural y el **49,6 %** tiene causa en investigación. La mayor participación de motocicletas y conductores sin licencia ya formaba parte de la definición del grupo; **las clases y la localización aportan el contraste nuevo**.

## 6. Análisis espacial con GeoPandas

Siguiendo el laboratorio de GeoPandas, convertimos **longitud → X**, **latitud → Y** y declaramos `EPSG:4326` (coordenadas geográficas). Cada punto conserva la etiqueta K-means asignada por atributos. El contorno `peru_contorno.geojson` permite ubicar los registros en el territorio nacional.

In [ ]:
gdf = gpd.GeoDataFrame(
    df, geometry=gpd.points_from_xy(df["coordenadas_longitud"],
                                    df["coordenadas_latitud"]), crs="EPSG:4326")
from pathlib import Path
ruta_mapa = "peru_shapes/per_admbnda_adm1_ign_20200714.shp"
peru = gpd.read_file(ruta_mapa if Path(ruta_mapa).exists() else "peru_contorno.geojson").to_crs(gdf.crs)
nombres = {0: "G0 · Mayor gravedad", 1: "G1 · Fuga vehicular",
           2: "G2 · Seguro y CITV", 3: "G3 · Motos y licencia"}
colores = {0: "#c0392b", 1: "#e69f00", 2: "#2874a6", 3: "#239b56"}

fig, ax = plt.subplots(figsize=(9, 9))
peru.plot(ax=ax, color="#f5f5f5", edgecolor="#8c8c8c", linewidth=.7)
for k in range(4):
    pts = gdf[gdf["cluster_kmeans"] == k]
    pts.plot(ax=ax, color=colores[k], markersize=3, alpha=.55,
             label=f"{nombres[k]} ({len(pts):,})")
ax.legend(loc="lower left", fontsize=9, markerscale=3, frameon=True)
ax.set_title("Siniestros fatales: clústeres por atributos en el mapa")
ax.set_axis_off()
plt.tight_layout()
plt.savefig("mapa_4_clusters_peru.png", dpi=160)
plt.show()

In [ ]:
# Un mapa por perfil evita que los puntos de un grupo oculten a otro.
fig, axes = plt.subplots(2, 2, figsize=(11, 13))
for k, ax in enumerate(axes.flat):
    peru.plot(ax=ax, color="#fafafa", edgecolor="#999999", linewidth=.6)
    gdf[gdf["cluster_kmeans"] == k].plot(
        ax=ax, color=colores[k], markersize=4, alpha=.55)
    ax.set_title(f"{nombres[k]} · n = {(df.cluster_kmeans == k).sum():,}")
    ax.set_axis_off()
plt.tight_layout()
plt.savefig("mapas_por_grupo.png", dpi=150)
plt.show()

### Comparación espacial cuantitativa

Para ir más allá del mapa, dividimos las coordenadas en tres **franjas de latitud**: sur (latitud < −14°), centro (−14° a −8°) y norte (> −8°). Son límites descriptivos y aproximados, **no departamentos ni regiones administrativas**. La tabla muestra el porcentaje de registros de **cada clúster** dentro de esas franjas.

In [ ]:
franjas = pd.cut(df["coordenadas_latitud"], bins=[-19, -14, -8, -3],
                 labels=["Sur (< -14°)", "Centro (-14° a -8°)", "Norte (> -8°)"])
distribucion = pd.crosstab(df["cluster_kmeans"], franjas, normalize="index") * 100
display(distribucion.round(1))

### Distribución de perfiles por departamento o distrito

Los siguientes mapas muestran **qué porcentaje de los siniestros fatales registrados en cada territorio pertenece a cada clúster**, no el porcentaje de accidentes nacionales que ocurrieron allí. Se utilizan los límites administrativos del laboratorio del lunes. Si no están disponibles, esta sección se omite sin afectar K-means.

In [ ]:
# Coropléticos adicionales por departamento o distrito (requiere peru_shapes)
from pathlib import Path

nivel = "distrito"  # también puede ser "region"
rutas = {
    "region": ("peru_shapes/per_admbnda_adm1_ign_20200714.shp", "ADM1_PCODE"),
    "distrito": ("peru_shapes/per_admbnda_adm3_ign_20200714.shp", "ADM3_PCODE")
}
ruta, codigo = rutas[nivel]

if Path(ruta).exists():
    zonas = gpd.read_file(ruta).to_crs(gdf.crs)
    datos = gpd.sjoin(gdf[["cluster_kmeans", "geometry"]],
                      zonas[[codigo, "geometry"]], how="inner", predicate="within")
    conteos = pd.crosstab(datos[codigo], datos["cluster_kmeans"])
    conteos = conteos.reindex(columns=range(4), fill_value=0)
    porcentajes = conteos.div(conteos.sum(axis=1), axis=0).mul(100)
    porcentajes.columns = [f"G{k}" for k in range(4)]
    mapa = zonas.merge(porcentajes, left_on=codigo, right_index=True, how="left")

    fig_territorial, axes = plt.subplots(2, 2, figsize=(13, 13))
    for k, ax in enumerate(axes.flat):
        mapa.plot(column=f"G{k}", ax=ax, cmap="YlOrRd", vmin=0, vmax=100,
                  edgecolor="gray", linewidth=.2, legend=True,
                  legend_kwds={"label": "% de siniestros", "shrink": .65},
                  missing_kwds={"color": "lightgray"})
        ax.set_title(nombres[k])
        ax.set_axis_off()
    plt.suptitle(f"Composición de los siniestros por {nivel}", fontsize=15)
    plt.tight_layout()
    fig_territorial.savefig(f"mapa_clusters_por_{nivel}.png", dpi=250, bbox_inches="tight")
    plt.show()
else:
    print("Sin shapefile administrativo: se omite el mapa territorial. K-means y los mapas de puntos no cambian.")

El análisis espacial de los clústeres revela diferencias territoriales en los perfiles de siniestros fatales del Perú. 

1. Los accidentes de mayor gravedad humana se encuentran predominantemente en zonas rurales
2. El perfil caracterizado por motocicletas y conductores sin licencia vigente presenta una mayor participación relativa en el norte del país.
3. Siniestros asociados con fugas vehiculares y atropellos se distribuyen de manera semejante entre zonas urbanas y rurales*.

### Distritos donde cada perfil tiene mayor representación

Se muestra el **porcentaje de siniestros del distrito** que pertenece a cada grupo, con un mínimo de diez siniestros por distrito. La proporción nacional sirve de referencia; un porcentaje alto con pocos casos debe interpretarse con cautela.

In [ ]:
# Ranking distrital: se ejecuta solo si están los límites distritales
ruta_distritos = "peru_shapes/per_admbnda_adm3_ign_20200714.shp"
if Path(ruta_distritos).exists():
    distritos = gpd.read_file(ruta_distritos).to_crs(gdf.crs)
    datos_distrito = gpd.sjoin(gdf[["cluster_kmeans", "geometry"]],
                              distritos[["ADM3_PCODE", "geometry"]],
                              how="inner", predicate="within")
    conteos_distrito = pd.crosstab(datos_distrito["ADM3_PCODE"],
                                  datos_distrito["cluster_kmeans"]).reindex(columns=range(4), fill_value=0)
    tabla = distritos[["ADM1_ES", "ADM2_ES", "ADM3_ES", "ADM3_PCODE"]].copy()
    tabla["Total"] = tabla["ADM3_PCODE"].map(conteos_distrito.sum(axis=1)).fillna(0).astype(int)
    referencia = df["cluster_kmeans"].value_counts(normalize=True).mul(100)
    for k in range(4):
        tabla[f"Casos_G{k}"] = tabla["ADM3_PCODE"].map(conteos_distrito[k]).fillna(0).astype(int)
        tabla[f"Porcentaje_G{k}"] = 100 * tabla[f"Casos_G{k}"] / tabla["Total"].replace(0, np.nan)
        ranking = (tabla[tabla["Total"] >= 10]
                   .sort_values([f"Porcentaje_G{k}", f"Casos_G{k}"], ascending=False)
                   .head(10)[["ADM1_ES", "ADM2_ES", "ADM3_ES", "Total",
                               f"Casos_G{k}", f"Porcentaje_G{k}"]])
        print(f"\n{nombres[k]} | Referencia nacional: {referencia[k]:.1f}%")
        display(ranking.round(1))
else:
    print("Ranking distrital omitido: falta el shapefile ADM3 de peru_shapes.")

**Posibles líneas de intervención (hipótesis, no efectos demostrados):**

- **G0 — Mayor gravedad:** inspeccionar condiciones viales rurales, velocidad, barreras de contención y tiempos de atención de emergencias.
- **G1 — Fuga y atropellos:** estudiar cruces peatonales, iluminación, velocidad e identificación de vehículos fugados.
- **G2 — Seguro/CITV:** investigar causas, vías y horarios; el alto cumplimiento documental registrado no explica por sí solo la ocurrencia de accidentes.
- **G3 — Motos y licencias:** revisar uso de casco, condiciones de circulación, licencias y seguridad de intersecciones.

Priorizar distritos combinando **porcentaje, cantidad absoluta de siniestros y víctimas**, no únicamente los más rojos. Estos datos describen siniestros *fatales*, no tasas de riesgo por vehículo o población.

**NOTA:** G3 tiene mayor presencia relativa en la franja norte (**33,9 %** de sus siniestros) que G2 (**16,8 %**). G0 y G1 se sitúan entre ambos; en los **cuatro grupos** predomina la franja central según estos límites. Además, G0 es mayoritariamente rural (82,5 %), mientras G1 tiene un reparto rural/urbano cercano al 50 %.

### Todo se resume como: 

| Perfil | Rasgo que lo definió en K-means | Hallazgo posterior (variables excluidas) |
|---|---|---|
| **G0** | Fallecidos y lesionados elevados | Predominan choques y despistes rurales. |
| **G1** | Elevada proporción de vehículos fugados | 92,2 % son atropellos; 92,3 % tiene peatón involucrado. |
| **G2** | Mayor cobertura registrada de seguro y CITV | Combina choques, atropellos y despistes sin una clase dominante. |
| **G3** | Mayor frecuencia de motos y falta de licencia | Predominan choques y despistes; mayor participación relativa en la franja norte. |

Los resultados son exploratorios: **identifican patrones entre siniestros fatales**, pero no establecen causas de accidentes ni riesgos poblacionales.

## 7. Mapas interactivos complementarios (Folium)

Estos mapas **no cambian el clustering**. El primero permite acercarse a los puntos originales y consultar su etiqueta; los siguientes muestran la **cantidad absoluta** de siniestros por departamento y dentro del departamento con más registros. Esto complementa el mapa de porcentajes anterior, que responde a otra pregunta.

Los mapas interactivos se muestran **directamente dentro del notebook**, como en el Hito 3 original, y también se guardan como archivos HTML. Esto evita problemas de rutas relativas al usar `IFrame` en JupyterLab o VS Code. Si el fondo base no carga, se necesita conexión a Internet; las capas con los datos siguen siendo independientes del proveedor de mosaicos.

### 7.1. Mapa interactivo de perfiles K-means

Los marcadores se agrupan automáticamente según el zoom. Se muestran **todos** los siniestros, sin alterar los cuatro grupos calculados. `FastMarkerCluster` reduce el peso del mapa.


In [ ]:
# Mapa final
map_final = folium.Map(location=[-9.2, -75], zoom_start=5, tiles="Esri.WorldGrayCanvas")

# Mapa interactivo de los cuatro perfiles, con todos los siniestros
mapa_puntos = folium.Map(location=[-9.2, -75], zoom_start=5, tiles="Esri.WorldGrayCanvas")
capa_puntos = folium.FeatureGroup(name="Siniestros", show=True)
for k in range(4):
    capa = folium.FeatureGroup(name=nombres[k]).add_to(mapa_puntos)
    posiciones = gdf.loc[gdf["cluster_kmeans"] == k,
                          ["coordenadas_latitud", "coordenadas_longitud"]].values.tolist()
    marcador = ("function(p) { return L.circleMarker([p[0], p[1]], "
                "{radius:3, color:'%s', fillColor:'%s', fillOpacity:0.8})"
                ".bindPopup('%s'); }") % (colores[k], colores[k], nombres[k])
    FastMarkerCluster(posiciones, callback=marcador).add_to(capa_puntos)
    FastMarkerCluster(posiciones, callback=marcador).add_to(capa)
folium.LayerControl().add_to(mapa_puntos)
capa_puntos.add_to(map_final)
mapa_puntos.save("mapa_kmeans_interactivo.html")
display(mapa_puntos)

### 7.2. Total de siniestros por departamento y detalle distrital

A diferencia de los cuatro coropléticos anteriores, aquí el color indica el **número de siniestros fatales registrados**, sin separar los clústeres. Así distinguimos un porcentaje alto de un volumen alto de casos. Requiere los shapefiles `ADM1` y `ADM3` utilizados en el Hito 3.

In [ ]:
# Conteos reales por departamento (ADM1)
ruta_region = "peru_shapes/per_admbnda_adm1_ign_20200714.shp"
if Path(ruta_region).exists():
    regiones = gpd.read_file(ruta_region).to_crs(gdf.crs)
    puntos_region = gpd.sjoin(gdf[["geometry"]],
                              regiones[["ADM1_PCODE", "geometry"]],
                              how="inner", predicate="within")
    total_region = puntos_region.groupby("ADM1_PCODE").size()
    regiones["n_siniestros"] = regiones["ADM1_PCODE"].map(total_region).fillna(0).astype(int)
    maximo = max(int(regiones["n_siniestros"].max()), 1)
    paleta = linear.YlOrRd_09.scale(0, maximo)
    mapa_regiones = folium.Map(location=[-9.2, -75], zoom_start=5, tiles="Esri.WorldGrayCanvas")
    folium.GeoJson(
        regiones[["ADM1_ES", "n_siniestros", "geometry"]].to_json(),
        style_function=lambda feat: {"fillColor": paleta(feat["properties"]["n_siniestros"]),
                                     "color": "#888", "weight": .5, "fillOpacity": .75},
        tooltip=folium.GeoJsonTooltip(fields=["ADM1_ES", "n_siniestros"],
                                       aliases=["Departamento", "Siniestros fatales"])
    ).add_to(mapa_regiones)
    capa_regiones = folium.FeatureGroup(name="Regiones", show=False)
    folium.GeoJson(
            regiones[["ADM1_ES", "n_siniestros", "geometry"]].to_json(),
            style_function=lambda feat: {"fillColor": paleta(feat["properties"]["n_siniestros"]),
                                         "color": "#888", "weight": .5, "fillOpacity": .75},
            tooltip=folium.GeoJsonTooltip(fields=["ADM1_ES", "n_siniestros"],
                                           aliases=["Departamento", "Siniestros fatales"])
        ).add_to(capa_regiones)
    capa_regiones.add_to(map_final)
    paleta.caption = "Cantidad de siniestros fatales registrados"
    paleta.add_to(mapa_regiones)
    limites = regiones.total_bounds  # oeste, sur, este, norte
    mapa_regiones.fit_bounds([[limites[1], limites[0]], [limites[3], limites[2]]])
    mapa_regiones.save("mapa_total_departamentos.html")
    display(mapa_regiones)
else:
    print("Mapa de conteos por departamento omitido: falta ADM1.")

In [ ]:
# Detalle por distritos del departamento con más siniestros registrados
if Path(ruta_region).exists() and Path(ruta_distritos).exists():
    reg_top = regiones.loc[regiones["n_siniestros"].idxmax()]
    distrito_top = gpd.read_file(ruta_distritos).to_crs(gdf.crs)
    distrito_top = distrito_top[distrito_top["ADM1_ES"] == reg_top["ADM1_ES"]].copy()
    puntos_top = gpd.sjoin(gdf[["geometry"]],
                           distrito_top[["ADM3_PCODE", "geometry"]],
                           how="inner", predicate="within")
    por_distrito = puntos_top.groupby("ADM3_PCODE").size()
    distrito_top["n_siniestros"] = distrito_top["ADM3_PCODE"].map(por_distrito).fillna(0).astype(int)
    b = distrito_top.total_bounds  # lon_min, lat_min, lon_max, lat_max
    mapa_distritos = folium.Map(location=[(b[1]+b[3])/2, (b[0]+b[2])/2],
                                zoom_start=8, tiles="Esri.WorldGrayCanvas")
    paleta_d = linear.YlOrRd_09.scale(0, max(int(distrito_top["n_siniestros"].max()), 1))
    folium.GeoJson(
        distrito_top[["ADM3_ES", "n_siniestros", "geometry"]].to_json(),
        style_function=lambda feat: {"fillColor": paleta_d(feat["properties"]["n_siniestros"]),
                                     "color": "#888", "weight": .5, "fillOpacity": .8},
        tooltip=folium.GeoJsonTooltip(fields=["ADM3_ES", "n_siniestros"],
                                       aliases=["Distrito", "Siniestros fatales"])
    ).add_to(mapa_distritos)
    capa_distrito = folium.FeatureGroup(name="Distritos (Región con más siniestros)", show=False)
    folium.GeoJson(
            distrito_top[["ADM3_ES", "n_siniestros", "geometry"]].to_json(),
            style_function=lambda feat: {"fillColor": paleta_d(feat["properties"]["n_siniestros"]),
                                         "color": "#888", "weight": .5, "fillOpacity": .8},
            tooltip=folium.GeoJsonTooltip(fields=["ADM3_ES", "n_siniestros"],
                                           aliases=["Distrito", "Siniestros fatales"])
    ).add_to(capa_distrito)
    capa_distrito.add_to(map_final)
    paleta_d.caption = f"Número de siniestros: {reg_top['ADM1_ES']}"
    paleta_d.add_to(mapa_distritos)
    mapa_distritos.fit_bounds([[b[1], b[0]], [b[3], b[2]]])
    mapa_distritos.save("mapa_total_distritos_region_top.html")
    print("Departamento con más siniestros registrados:", reg_top["ADM1_ES"])
    display(mapa_distritos)
else:
    print("Detalle de distritos omitido: faltan ADM1 y/o ADM3.")

## 8. DBSCAN espacial como comparación exploratoria

DBSCAN encuentra grupos por cercanía geográfica, no por similitud de atributos. Es un análisis **complementario** y no modifica las etiquetas K-means. Para el mapa se utiliza una sola capa GeoJSON con símbolos por grupo, más ligera que miles de marcadores individuales.

In [ ]:
# Distancia al vecino número 50: orientación para elegir el radio espacial
coords_rad = np.radians(gdf[["coordenadas_latitud", "coordenadas_longitud"]].to_numpy())
vecinos = NearestNeighbors(n_neighbors=50, metric="haversine", algorithm="ball_tree").fit(coords_rad)
distancias, _ = vecinos.kneighbors(coords_rad)
k_distancias = np.sort(distancias[:, -1] * 6371.0088)  # kilómetros
plt.figure(figsize=(8, 3))
plt.plot(k_distancias)
plt.title("Distancia al vecino 50")
plt.xlabel("Observaciones ordenadas")
plt.ylabel("Distancia (km)")
plt.grid(alpha=.3)
plt.show()

In [ ]:
# Comparación de radios para DBSCAN espacial (distancia Haversine, en kilómetros)
resumen_dbscan = []
for eps_km in [40, 45, 50, 55, 60]:
    etiquetas_db = DBSCAN(eps=eps_km / 6371.0088, min_samples=50,
                          metric="haversine", algorithm="ball_tree").fit_predict(coords_rad)
    n_grupos = len(set(etiquetas_db) - {-1})
    resumen_dbscan.append([eps_km, n_grupos, (etiquetas_db == -1).sum()])
display(pd.DataFrame(resumen_dbscan, columns=["Radio (km)", "Grupos espaciales", "Puntos ruido"]))

# Mapa complementario para radio = 45 km
labels_db = DBSCAN(eps=45 / 6371.0088, min_samples=50,
                   metric="haversine", algorithm="ball_tree").fit_predict(coords_rad)
gdf["cluster_dbscan"] = labels_db
paleta_db = ["#2874a6", "#239b56", "#c0392b", "#9b59b6", "#e69f00", "#16a085"]
mapa_db = folium.Map(location=[-9.2, -75], zoom_start=5, tiles="Esri.WorldGrayCanvas")
folium.GeoJson(
    gdf[["cluster_dbscan", "geometry"]].to_json(),
    marker=folium.CircleMarker(radius=2, weight=0.5, fill=True, fill_opacity=0.7),
    style_function=lambda f: {
        "color": "#999999" if f["properties"]["cluster_dbscan"] == -1
                 else paleta_db[f["properties"]["cluster_dbscan"] % len(paleta_db)],
        "fillColor": "#999999" if f["properties"]["cluster_dbscan"] == -1
                     else paleta_db[f["properties"]["cluster_dbscan"] % len(paleta_db)]
    },
    tooltip=folium.GeoJsonTooltip(fields=["cluster_dbscan"], aliases=["Grupo DBSCAN"])
).add_to(mapa_db)
capa_dbscan = folium.FeatureGroup(name="DBSCAN", show=False)
folium.GeoJson(
    gdf[["cluster_dbscan", "geometry"]].to_json(),
    marker=folium.CircleMarker(radius=2, weight=0.5, fill=True, fill_opacity=0.7),
    style_function=lambda f: {
        "color": "#999999" if f["properties"]["cluster_dbscan"] == -1
                 else paleta_db[f["properties"]["cluster_dbscan"] % len(paleta_db)],
        "fillColor": "#999999" if f["properties"]["cluster_dbscan"] == -1
                     else paleta_db[f["properties"]["cluster_dbscan"] % len(paleta_db)]
    },
    tooltip=folium.GeoJsonTooltip(fields=["cluster_dbscan"], aliases=["Grupo DBSCAN"])
).add_to(capa_dbscan)
capa_dbscan.add_to(map_final)
mapa_db.save("mapa_dbscan_espacial.html")
display(mapa_db)

### Mapa final

In [ ]:
folium.LayerControl().add_to(map_final)
map_final.save("mapa_final.html")
map_final

## 9. Guardar resultados

Se exporta la etiqueta de K-means, el GeoJSON de los puntos y los mapas estáticos e interactivos que se hayan generado. **La columna DBSCAN se mantiene separada** y no reemplaza las etiquetas de los cuatro perfiles.

In [ ]:
df.to_csv("siniestros_kmeans_4grupos.csv", index=False)
gdf[["cluster_kmeans", "cantidad_de_fallecidos", "geometry"]].to_file(
    "siniestros_kmeans_4grupos.geojson", driver="GeoJSON")
print("Guardados: CSV, GeoJSON, mapas PNG e interactivos HTML (cuando corresponde).")